<div style="font-family:system-ui;border-left:6px solid #B8721A;padding:6px 16px">
<div style="font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#566273">The AI-Ready Organisation · Lab 1 · Module 1 · ≈25 min</div>
<div style="font-size:26px;font-weight:800;margin:6px 0">Predictive versus generative in 20 minutes</div>
<div style="font-size:15px"><b>You will:</b> score customer churn risk with a predictive model, then use a generative model to draft a retention offer, and compare what each kind of AI is good for.</div>
<div style="font-size:13px;color:#566273;margin-top:6px">Dataset: IBM Telco Customer Churn (7,043 customers, public sample data)</div>
</div>

**How to use this notebook** · No coding needed.
1. In the menu choose **Runtime → Run all** (in Kaggle: **Run All**). The first run takes 1–3 minutes.
2. Change values in the **form fields** on the right of each cell (sliders, dropdowns). The cell reruns when you change a value; if it does not, press the play button.
3. Read the plain-English findings under each chart and record your answers in the final cell.

If the real dataset cannot be downloaded (for example, without a Kaggle sign-in), the notebook automatically switches to a **realistic simulated copy with the same structure** and tells you so. The lessons still hold.

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import warnings, os, io, zipfile, urllib.request
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
plt.rcParams.update({"figure.figsize": (9, 4.4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 11, "axes.titleweight": "bold"})
BUILD, BUY, BLEND, INK, GOOD, BAD = "#B8721A", "#13807B", "#4255B0", "#111822", "#2D7A4C", "#B03A3A"
pd.set_option("display.max_columns", 30)

def say(text):
    display(Markdown(text))

def scorecard(title, rows):
    """rows: list of (label, value, note)"""
    body = "".join(f"<tr><td style='padding:6px 12px'>{a}</td><td style='padding:6px 12px;font-weight:700;text-align:right'>{b}</td>"
                   f"<td style='padding:6px 12px;color:#566273'>{c}</td></tr>" for a, b, c in rows)
    display(HTML(f"<div style='font-family:system-ui;margin:8px 0'><div style='font-weight:700;font-size:15px;margin-bottom:4px'>{title}</div>"
                 f"<table style='border-collapse:collapse;font-size:14px'>{body}</table></div>"))

def try_kagglehub(handle, filename):
    """Download a public Kaggle dataset. Returns a file path or None."""
    try:
        try:
            import kagglehub
        except ImportError:
            os.system("pip -q install kagglehub")
            import kagglehub
        path = kagglehub.dataset_download(handle)
        for root, _, files in os.walk(path):
            if filename in files:
                return os.path.join(root, filename)
    except Exception as e:
        print(f"Kaggle download unavailable ({type(e).__name__}).")
    return None

RNG = np.random.default_rng(42)
DATA_NOTE = ""

## 1 · Load the customer data
Each row is a customer of a fictional telecoms company: contract type, services, monthly bill, and whether they left (churned).

In [ ]:
#@title Load data { display-mode: "form" }
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
try:
    df = pd.read_csv(URL)
    DATA_NOTE = "Real IBM Telco Customer Churn data."
except Exception:
    n = 7043
    tenure = RNG.integers(0, 73, n)
    contract = RNG.choice(["Month-to-month", "One year", "Two year"], n, p=[.55, .21, .24])
    internet = RNG.choice(["DSL", "Fiber optic", "No"], n, p=[.34, .44, .22])
    monthly = np.round(np.where(internet == "Fiber optic", 85, np.where(internet == "DSL", 55, 22)) + RNG.normal(0, 12, n), 2)
    logit = -1.2 + 1.6*(contract == "Month-to-month") - 0.04*tenure + 0.8*(internet == "Fiber optic") + RNG.normal(0, .6, n)
    df = pd.DataFrame({"customerID": [f"SIM-{i:05d}" for i in range(n)], "tenure": tenure, "Contract": contract,
        "InternetService": internet, "MonthlyCharges": monthly, "TotalCharges": monthly*np.maximum(tenure, 1),
        "SeniorCitizen": RNG.binomial(1, .16, n), "PaperlessBilling": RNG.choice(["Yes", "No"], n),
        "PaymentMethod": RNG.choice(["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"], n),
        "TechSupport": RNG.choice(["Yes", "No", "No internet service"], n), "OnlineSecurity": RNG.choice(["Yes", "No", "No internet service"], n),
        "Churn": np.where(RNG.random(n) < 1/(1+np.exp(-logit)), "Yes", "No")})
    DATA_NOTE = "Simulated copy (download failed). Same columns as the IBM dataset."
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["churned"] = (df["Churn"] == "Yes").astype(int)
say(f"**{DATA_NOTE}** {len(df):,} customers, of whom **{df.churned.mean():.0%} churned**.")
display(df.head(5))

## 2 · Train a predictive model (already written for you)
A logistic regression learns which customer attributes are associated with leaving. Its output is a **score between 0 and 1**, not words.

In [ ]:
#@title Train the churn model { display-mode: "form" }
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
num = ["tenure", "MonthlyCharges", "SeniorCitizen"]
cat = ["Contract", "InternetService", "PaymentMethod", "TechSupport", "OnlineSecurity", "PaperlessBilling"]
X, y = df[num + cat], df["churned"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.3, random_state=7, stratify=y)
model = make_pipeline(ColumnTransformer([("n", StandardScaler(), num), ("c", OneHotEncoder(handle_unknown="ignore"), cat)]),
                      LogisticRegression(max_iter=1000))
model.fit(Xtr, ytr)
df["risk"] = model.predict_proba(X)[:, 1]
test = df.loc[Xte.index]
auc = roc_auc_score(yte, test["risk"])
say(f"Model trained. On customers it has never seen, it ranks a random leaver above a random stayer **{auc:.0%} of the time** (AUC {auc:.2f}). 50% would be guessing.")

## 3 · Set the risk threshold
Move the slider. Customers scoring above it go to the retention team. Watch the trade-off: a lower threshold finds more leavers but sends more loyal customers too.

In [ ]:
#@title Choose a threshold { display-mode: "form" }
risk_threshold = 0.5 #@param {type:"slider", min:0.2, max:0.8, step:0.05}
flag = test["risk"] >= risk_threshold
caught, flagged = int((flag & (test.churned == 1)).sum()), int(flag.sum())
precision = caught / max(flagged, 1); recall = caught / max(int(test.churned.sum()), 1)
rev = (df.loc[df.risk >= risk_threshold] .query("churned == 1")["MonthlyCharges"].sum()) * 12
fig, ax = plt.subplots()
ax.hist(test.loc[test.churned == 0, "risk"], bins=30, alpha=.6, color=BUY, label="Stayed")
ax.hist(test.loc[test.churned == 1, "risk"], bins=30, alpha=.6, color=BUILD, label="Left")
ax.axvline(risk_threshold, color=INK, ls="--", lw=2); ax.text(risk_threshold+.01, ax.get_ylim()[1]*.9, "threshold")
ax.set(title="Churn risk scores on unseen customers", xlabel="Predicted risk", ylabel="Customers"); ax.legend(); plt.show()
scorecard("At this threshold", [
    ("Customers sent to retention", f"{flagged:,}", "in the test group"),
    ("Of them, real leavers (precision)", f"{precision:.0%}", "the rest are false alarms"),
    ("Share of all leavers found (recall)", f"{recall:.0%}", "the rest leave unnoticed"),
    ("Annual revenue of flagged leavers (all customers)", f"${rev:,.0f}", "what a perfect save would protect"),
])

## 4 · Who is in the at-risk segment?

In [ ]:
#@title Profile the at-risk segment { display-mode: "form" }
seg = df[df.risk >= risk_threshold]
prof = pd.DataFrame({
    "At-risk segment": [len(seg), seg.tenure.median(), seg.MonthlyCharges.mean(), (seg.Contract == "Month-to-month").mean()],
    "All customers": [len(df), df.tenure.median(), df.MonthlyCharges.mean(), (df.Contract == "Month-to-month").mean()],
}, index=["Customers", "Median tenure (months)", "Average monthly bill ($)", "Share on month-to-month"]).round(2)
display(prof)
top_contract = seg.Contract.mode()[0]; top_net = seg.InternetService.mode()[0]
say(f"Typical at-risk customer: **{top_contract}** contract, **{top_net}** internet, about **{seg.tenure.median():.0f} months** with the company, paying around **${seg.MonthlyCharges.mean():.0f}** a month.")

## 5 · Hand the segment to a generative model
The predictive model told us **who**. Now a generative model drafts **what to say**. In Colab this cell uses Colab's built-in AI if it is available on your account; otherwise it shows a template draft so you can still compare.

In [ ]:
#@title Draft a retention offer { display-mode: "form" }
offer_tone = "Warm and personal" #@param ["Warm and personal", "Short and direct", "Value-focused"]
offer = "12-month contract with 20% off for 6 months" #@param {type:"string"}
prompt = (f"Write a 90-word retention email for a telecom customer on a {top_contract} contract with {top_net} internet, "
          f"about {seg.tenure.median():.0f} months tenure, paying about ${seg.MonthlyCharges.mean():.0f} per month. "
          f"Tone: {offer_tone}. Offer: {offer}. No exaggerated claims.")
draft = None
try:
    from google.colab import ai
    draft = ai.generate_text(prompt)
    source = "Generated by Colab's built-in AI"
except Exception:
    source = "Template draft (generative AI not available in this environment)"
    draft = (f"Subject: A better plan for you\n\nHi there,\n\nThank you for being with us for {seg.tenure.median():.0f} months. "
             f"We know your {top_net.lower()} service matters to you, so we'd like to offer you a {offer.lower()}. "
             "It keeps the service you use and lowers your bill. Reply to this email or call us and we'll switch you over in minutes.\n\nThe Customer Team")
say(f"**Prompt sent:** {prompt}")
say(f"**{source}:**")
print(draft)

## 6 · Compare the two kinds of AI
| | Predictive model | Generative model |
|---|---|---|
| Output | A score for every customer | A draft message |
| Learns from | Your historical data | Pre-trained on public text; guided by your prompt |
| Can be checked by | Comparing to what actually happened | A human reading it |
| Main risk | Drift as customers change | Fluent but wrong or off-brand text |
| Best used for | Deciding **who** and **how much** | Producing **content** at scale |

## Your reflection → capstone §1
Answer in the form fields, then run the cell. Copy the summary into §1 of your business case on the course page.

In [ ]:
#@title Record your answers { display-mode: "form" }
q1 = "" #@param {type:"string"}
q2 = "" #@param {type:"string"}
q3 = "" #@param {type:"string"}
answers = [("One process in my organisation that needs a predictive score:", q1), ("One process that needs generated content:", q2), ("One process that needs both, like this lab:", q3)]
say("### Lab 1 reflection")
for q, a in answers:
    say(f"**{q}**  \n{a if a.strip() else '_(not answered yet)_'}")